# H1 Zero Trust Analysis

**H1: Zero Trust implementation improves anomaly detection performance in RFID baggage tracking systems compared to traditional rule based methods.**

**RQ1: How can Zero Trust improve anomaly detection in RFID baggage tracking systems?**

This notebook evaluates H1 by comparing traditional rule review with Zero Trust validation using the prepared RFID baggage tracking data.

**BLUF:** The analysis examines whether Zero Trust improves the ability to identify abnormal RFID baggage tracking activity when compared with traditional rule review.

## Step 1: Connect and Review the Prepared RFID Data

The prepared RFID baggage tracking file was connected from the Praxis project folder. The data was reviewed to confirm the number of scan records, baggage journeys, and scan labels before the H1 comparison continued.

In [2]:

# Step 1: Connect and review the prepared RFID data

# Load the libraries
import pandas as pd
import os

# Find the GitHub project folder
main_folder = os.getcwd()

# Check if the notebook is running from a subfolder
if not os.path.isdir(main_folder + "/06_Prepared_Data"):
    main_folder = os.path.dirname(main_folder)

# Set the prepared data folder
prepared_folder = main_folder + "/06_Prepared_Data"

# Set the prepared RFID file
prepared_file = prepared_folder + "/02_Prepared_RFID_Data.csv"

# Confirm the file can be found
print("Prepared RFID file:", prepared_file)
print("File found:", os.path.isfile(prepared_file))

# Load the prepared RFID data
h1_data = pd.read_csv(prepared_file)

# Check the number of records
print("Total RFID records:", len(h1_data))

# Check the number of columns
print("Total columns:", len(h1_data.columns))


Prepared RFID file: /workspaces/Aviation-Baggage-Cybersecurity-Research/06_Prepared_Data/02_Prepared_RFID_Data.csv
File found: True
Total RFID records: 31575
Total columns: 34


## Step 2: Apply the Traditional Rule Review

The traditional rule review provides the baseline for H1. Each scan is checked for required information, a recognized RFID reader, the expected location, and the expected custody sequence.

The traditional review does not use the stronger Zero Trust checks for reader authorization, timing, duplicate activity, or continued trust.

In [3]:
# Step 2: Apply the traditional rule review

# Make a copy of the prepared RFID data

traditional_data = h1_data.copy()

# Set the recognized RFID readers

recognized_readers = [
    "RDR001",
    "RDR002",
    "RDR003",
    "RDR004",
    "RDR005",
    "RDR006",
    "RDR007"
]

# Set the required fields

required_fields = [
    "bag_id",
    "tag_id",
    "reader_id",
    "reader_location",
    "custody_point",
    "timestamp",
    "flight_id",
    "route_id",
    "scan_id"
]

# Check for missing required information

traditional_data["required_information_present"] = (
    traditional_data[required_fields]
    .notna()
    .all(axis=1)
)

# Check whether the reader is recognized

traditional_data["recognized_reader"] = (
    traditional_data["reader_id"]
    .isin(recognized_readers)
)

# Check the recorded location

traditional_data["traditional_location_check"] = (
    traditional_data["location_match"]
)

# Check the custody sequence

traditional_data["traditional_sequence_check"] = (
    traditional_data["sequence_match"]
)

# Create the traditional rule decision

traditional_data["traditional_flagged"] = ~(
    traditional_data["required_information_present"]
    & traditional_data["recognized_reader"]
    & traditional_data["traditional_location_check"]
    & traditional_data["traditional_sequence_check"]
)

# Count the traditional rule decisions

print("Traditional rule review complete.")

print("\nTraditional rule decisions:")

print(
    traditional_data["traditional_flagged"]
    .value_counts()
)

# Show the number accepted and flagged

traditional_accepted = (
    traditional_data["traditional_flagged"] == False
).sum()

traditional_flagged = (
    traditional_data["traditional_flagged"] == True
).sum()

print(
    "\nTraditional accepted scans:",
    traditional_accepted
)

print(
    "Traditional flagged scans:",
    traditional_flagged
)

# Review each traditional rule check

print("\nRequired information check:")

print(
    traditional_data["required_information_present"]
    .value_counts()
)

print("\nRecognized reader check:")

print(
    traditional_data["recognized_reader"]
    .value_counts()
)

print("\nLocation check:")

print(
    traditional_data["traditional_location_check"]
    .value_counts()
)

print("\nSequence check:")

print(
    traditional_data["traditional_sequence_check"]
    .value_counts()
)

Traditional rule review complete.

Traditional rule decisions:
traditional_flagged
False    30750
True       825
Name: count, dtype: int64

Traditional accepted scans: 30750
Traditional flagged scans: 825

Required information check:
required_information_present
True    31575
Name: count, dtype: int64

Recognized reader check:
recognized_reader
True     31200
False      375
Name: count, dtype: int64

Location check:
traditional_location_check
1    31350
0      225
Name: count, dtype: int64

Sequence check:
traditional_sequence_check
1    31350
0      225
Name: count, dtype: int64


## Step 3: Apply the Zero Trust Review

The Zero Trust review adds stronger validation to the traditional rule checks. Each scan is reviewed for reader authorization, expected location, expected sequence, timing, and duplicate activity.

The known abnormal labels are not used to make the Zero Trust decision. They are kept for later comparison with the detection results.

In [4]:
# Step 3: Apply the Zero Trust review

# Make a copy of the prepared RFID data

zta_data = h1_data.copy()

# Change the timestamp to date and time format

zta_data["timestamp"] = pd.to_datetime(
    zta_data["timestamp"]
)

# Sort the scans by bag and time

zta_data = zta_data.sort_values(
    ["bag_id", "timestamp"]
).reset_index(drop=True)

# Check the required information

zta_data["zta_required_information"] = (
    zta_data[required_fields]
    .notna()
    .all(axis=1)
)

# Check whether the reader is authorized
# for the expected custody point

zta_data["authorized_reader"] = (
    zta_data["reader_id"]
    == zta_data["expected_reader_id"]
)

# Check the expected location

zta_data["zta_location_check"] = (
    zta_data["location_match"] == 1
)

# Check the expected custody sequence

zta_data["zta_sequence_check"] = (
    zta_data["sequence_match"] == 1
)

# Check whether the scan occurred
# within the expected time

zta_data["zta_timing_check"] = (
    zta_data["delayed_indicator"] == 0
)

# Get information from the previous scan

zta_data["previous_bag_id"] = (
    zta_data["bag_id"].shift(1)
)

zta_data["previous_reader_id"] = (
    zta_data["reader_id"].shift(1)
)

zta_data["previous_location"] = (
    zta_data["reader_location"].shift(1)
)

zta_data["previous_custody_point"] = (
    zta_data["custody_point"].shift(1)
)

# Check for a duplicate scan
# within 60 seconds

zta_data["duplicate_check"] = (
    (zta_data["bag_id"]
     == zta_data["previous_bag_id"])
    &
    (zta_data["reader_id"]
     == zta_data["previous_reader_id"])
    &
    (zta_data["reader_location"]
     == zta_data["previous_location"])
    &
    (zta_data["custody_point"]
     == zta_data["previous_custody_point"])
    &
    (zta_data["scan_gap_minutes"] <= 1)
)

# Create the Zero Trust decision

zta_data["zta_flagged"] = ~(
    zta_data["zta_required_information"]
    &
    zta_data["authorized_reader"]
    &
    zta_data["zta_location_check"]
    &
    zta_data["zta_sequence_check"]
    &
    zta_data["zta_timing_check"]
    &
    ~zta_data["duplicate_check"]
)

# Review the Zero Trust results

print("Zero Trust review complete.")

print("\nZero Trust decisions:")

print(
    zta_data["zta_flagged"]
    .value_counts()
)

# Count accepted and flagged scans

zta_accepted = (
    zta_data["zta_flagged"] == False
).sum()

zta_flagged = (
    zta_data["zta_flagged"] == True
).sum()

print(
    "\nZero Trust accepted scans:",
    zta_accepted
)

print(
    "Zero Trust flagged scans:",
    zta_flagged
)

# Review each Zero Trust check

print("\nAuthorized reader check:")

print(
    zta_data["authorized_reader"]
    .value_counts()
)

print("\nLocation check:")

print(
    zta_data["zta_location_check"]
    .value_counts()
)

print("\nSequence check:")

print(
    zta_data["zta_sequence_check"]
    .value_counts()
)

print("\nTiming check:")

print(
    zta_data["zta_timing_check"]
    .value_counts()
)

print("\nDuplicate scan check:")

print(
    zta_data["duplicate_check"]
    .value_counts()
)

Zero Trust review complete.

Zero Trust decisions:
zta_flagged
False    30300
True      1275
Name: count, dtype: int64

Zero Trust accepted scans: 30300
Zero Trust flagged scans: 1275

Authorized reader check:
authorized_reader
True     31200
False      375
Name: count, dtype: int64

Location check:
zta_location_check
True     31350
False      225
Name: count, dtype: int64

Sequence check:
zta_sequence_check
True     31350
False      225
Name: count, dtype: int64

Timing check:
zta_timing_check
True     31350
False      225
Name: count, dtype: int64

Duplicate scan check:
duplicate_check
False    31350
True       225
Name: count, dtype: int64


## Step 4: Compare Traditional and Zero Trust Performance

The Traditional Rule and Zero Trust results are compared using the same abnormal scan labels. Abnormal RFID activity is treated as the positive class.

Accuracy, precision, recall, F1 score, false positive rate, false negative rate, and confusion matrix values are calculated for both approaches.

In [5]:
# Step 4: Compare Traditional and Zero Trust performance

# Load the evaluation tools

from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

# ---------------------------------------------------------
# Traditional Rule results
# ---------------------------------------------------------

# Set the actual scan labels

traditional_actual = (
    traditional_data["final_label"] == "Abnormal"
)

# Set the Traditional Rule predictions

traditional_prediction = (
    traditional_data["traditional_flagged"] == True
)

# Create the Traditional Rule confusion matrix

traditional_matrix = confusion_matrix(
    traditional_actual,
    traditional_prediction
)

# Get the Traditional Rule values

traditional_tn = traditional_matrix[0, 0]
traditional_fp = traditional_matrix[0, 1]
traditional_fn = traditional_matrix[1, 0]
traditional_tp = traditional_matrix[1, 1]

# Calculate the Traditional Rule measures

traditional_accuracy = accuracy_score(
    traditional_actual,
    traditional_prediction
)

traditional_precision = precision_score(
    traditional_actual,
    traditional_prediction
)

traditional_recall = recall_score(
    traditional_actual,
    traditional_prediction
)

traditional_f1 = f1_score(
    traditional_actual,
    traditional_prediction
)

traditional_fpr = (
    traditional_fp /
    (traditional_fp + traditional_tn)
)

traditional_fnr = (
    traditional_fn /
    (traditional_fn + traditional_tp)
)

# ---------------------------------------------------------
# Zero Trust results
# ---------------------------------------------------------

# Set the actual scan labels

zta_actual = (
    zta_data["final_label"] == "Abnormal"
)

# Set the Zero Trust predictions

zta_prediction = (
    zta_data["zta_flagged"] == True
)

# Create the Zero Trust confusion matrix

zta_matrix = confusion_matrix(
    zta_actual,
    zta_prediction
)

# Get the Zero Trust values

zta_tn = zta_matrix[0, 0]
zta_fp = zta_matrix[0, 1]
zta_fn = zta_matrix[1, 0]
zta_tp = zta_matrix[1, 1]

# Calculate the Zero Trust measures

zta_accuracy = accuracy_score(
    zta_actual,
    zta_prediction
)

zta_precision = precision_score(
    zta_actual,
    zta_prediction
)

zta_recall = recall_score(
    zta_actual,
    zta_prediction
)

zta_f1 = f1_score(
    zta_actual,
    zta_prediction
)

zta_fpr = (
    zta_fp /
    (zta_fp + zta_tn)
)

zta_fnr = (
    zta_fn /
    (zta_fn + zta_tp)
)

# ---------------------------------------------------------
# Show the Traditional Rule results
# ---------------------------------------------------------

print("Traditional Rule Performance")

print("\nConfusion Matrix:")
print(traditional_matrix)

print("\nTrue Positives:", traditional_tp)
print("True Negatives:", traditional_tn)
print("False Positives:", traditional_fp)
print("False Negatives:", traditional_fn)

print("\nAccuracy:", round(traditional_accuracy, 4))
print("Precision:", round(traditional_precision, 4))
print("Recall:", round(traditional_recall, 4))
print("F1 Score:", round(traditional_f1, 4))
print("False Positive Rate:", round(traditional_fpr, 4))
print("False Negative Rate:", round(traditional_fnr, 4))

# ---------------------------------------------------------
# Show the Zero Trust results
# ---------------------------------------------------------

print("\nZero Trust Performance")

print("\nConfusion Matrix:")
print(zta_matrix)

print("\nTrue Positives:", zta_tp)
print("True Negatives:", zta_tn)
print("False Positives:", zta_fp)
print("False Negatives:", zta_fn)

print("\nAccuracy:", round(zta_accuracy, 4))
print("Precision:", round(zta_precision, 4))
print("Recall:", round(zta_recall, 4))
print("F1 Score:", round(zta_f1, 4))
print("False Positive Rate:", round(zta_fpr, 4))
print("False Negative Rate:", round(zta_fnr, 4))

Traditional Rule Performance

Confusion Matrix:
[[30225     0]
 [  525   825]]

True Positives: 825
True Negatives: 30225
False Positives: 0
False Negatives: 525

Accuracy: 0.9834
Precision: 1.0
Recall: 0.6111
F1 Score: 0.7586
False Positive Rate: 0.0
False Negative Rate: 0.3889

Zero Trust Performance

Confusion Matrix:
[[30225     0]
 [   75  1275]]

True Positives: 1275
True Negatives: 30225
False Positives: 0
False Negatives: 75

Accuracy: 0.9976
Precision: 1.0
Recall: 0.9444
F1 Score: 0.9714
False Positive Rate: 0.0
False Negative Rate: 0.0556


## Step 5: Validate Missing Scans and Save the H1 Results

Missing scans are evaluated at the baggage journey level because the missing event does not have a scan record in the prepared dataset.

Each journey is compared with its expected custody path. Direct routes require acceptance, screening, sorting, loading, arrival, and claim. Transfer routes also require a transfer event.

The missing scan results are reviewed separately from the scan level confusion matrices. Final Traditional Rule and Zero Trust performance results are then saved for the H1 comparison.

In [6]:
# Step 5: Validate missing scans and save the H1 results

# Set the expected direct route

direct_path = [
    "Acceptance",
    "Screening",
    "Sorting",
    "Loading",
    "Arrival",
    "Claim"
]

# Set the expected transfer route

transfer_path = [
    "Acceptance",
    "Screening",
    "Sorting",
    "Loading",
    "Transfer",
    "Arrival",
    "Claim"
]

# Create a list for the missing scan results

missing_results = []

# Review each baggage journey

for bag_id, bag_data in h1_data.groupby("bag_id"):

    # Get the route type

    route_type = bag_data["route_type"].iloc[0]

    # Set the expected custody path

    if route_type == "Direct":
        expected_path = direct_path
    else:
        expected_path = transfer_path

    # Get the recorded custody points

    recorded_points = (
        bag_data["custody_point"]
        .dropna()
        .tolist()
    )

    # Check for missing custody points

    missing_points = [
        point
        for point in expected_path
        if point not in recorded_points
    ]

    # Save the result

    missing_results.append({
        "bag_id": bag_id,
        "route_type": route_type,
        "missing_custody_detected": len(missing_points) > 0,
        "missing_custody_points": ", ".join(missing_points)
    })

# Create the missing scan results table

missing_data = pd.DataFrame(
    missing_results
)

# Add the actual missing scan label
# This is used only to check the detection result

journey_labels = (
    h1_data[
        ["bag_id", "journey_abnormal_type"]
    ]
    .drop_duplicates("bag_id")
)

missing_data = missing_data.merge(
    journey_labels,
    on="bag_id",
    how="left"
)

missing_data["actual_missing_scan"] = (
    missing_data["journey_abnormal_type"]
    == "Missing Scan"
)

# Show the missing scan results

print("Missing scan validation complete.")

print(
    "\nCustody paths flagged as incomplete:",
    missing_data["missing_custody_detected"].sum()
)

print(
    "Actual missing scan journeys:",
    missing_data["actual_missing_scan"].sum()
)

# Create the missing scan confusion matrix

missing_matrix = confusion_matrix(
    missing_data["actual_missing_scan"],
    missing_data["missing_custody_detected"]
)

missing_tn = missing_matrix[0, 0]
missing_fp = missing_matrix[0, 1]
missing_fn = missing_matrix[1, 0]
missing_tp = missing_matrix[1, 1]

print("\nMissing Scan Confusion Matrix:")
print(missing_matrix)

print("\nTrue Positives:", missing_tp)
print("True Negatives:", missing_tn)
print("False Positives:", missing_fp)
print("False Negatives:", missing_fn)

# ---------------------------------------------------------
# Create the H1 comparison table
# ---------------------------------------------------------

h1_comparison = pd.DataFrame({
    "Measure": [
        "True Positives",
        "True Negatives",
        "False Positives",
        "False Negatives",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "False Positive Rate",
        "False Negative Rate"
    ],

    "Traditional Rule": [
        traditional_tp,
        traditional_tn,
        traditional_fp,
        traditional_fn,
        traditional_accuracy,
        traditional_precision,
        traditional_recall,
        traditional_f1,
        traditional_fpr,
        traditional_fnr
    ],

    "Zero Trust": [
        zta_tp,
        zta_tn,
        zta_fp,
        zta_fn,
        zta_accuracy,
        zta_precision,
        zta_recall,
        zta_f1,
        zta_fpr,
        zta_fnr
    ]
})

print("\nH1 Traditional Rule and Zero Trust Comparison:")

print(
    h1_comparison.round(4)
)

# ---------------------------------------------------------
# Save the H1 results
# ---------------------------------------------------------

results_folder = (
    main_folder +
    "/07_H1_Results"
)

# Create the results folder

import os

os.makedirs(
    results_folder,
    exist_ok=True
)

# Save the Traditional Rule results

traditional_data.to_csv(
    results_folder +
    "/h1_traditional_rule_results.csv",
    index=False
)

# Save the Zero Trust results

zta_data.to_csv(
    results_folder +
    "/h1_zta_results.csv",
    index=False
)

# Save the comparison table

h1_comparison.to_csv(
    results_folder +
    "/h1_traditional_vs_zta_comparison.csv",
    index=False
)

# Save the missing scan results

missing_data.to_csv(
    results_folder +
    "/h1_missing_scan_results.csv",
    index=False
)

print("\nH1 result files saved successfully.")

Missing scan validation complete.

Custody paths flagged as incomplete: 150
Actual missing scan journeys: 150

Missing Scan Confusion Matrix:
[[4850    0]
 [   0  150]]

True Positives: 150
True Negatives: 4850
False Positives: 0
False Negatives: 0

H1 Traditional Rule and Zero Trust Comparison:
               Measure  Traditional Rule  Zero Trust
0       True Positives          825.0000   1275.0000
1       True Negatives        30225.0000  30225.0000
2      False Positives            0.0000      0.0000
3      False Negatives          525.0000     75.0000
4             Accuracy            0.9834      0.9976
5            Precision            1.0000      1.0000
6               Recall            0.6111      0.9444
7             F1 Score            0.7586      0.9714
8  False Positive Rate            0.0000      0.0000
9  False Negative Rate            0.3889      0.0556

H1 result files saved successfully.
